In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q16/q16_rewrite/checkpoints/pre_cell_3.pickle

In [4]:
%%RecordEvent
%%time
### cell 3 ###

# Optimized cell 3
# 1) Pre-compute the supplier keys to exclude via a single filter & drop_duplicates
sup_keys = (
    supplier['S_SUPPKEY']
    [supplier['S_COMMENT'].str.contains(r"Customer(\S|\s)*Complaints")]
    .drop_duplicates()
)

# 2) Filter parts by brand, type and size
part_filtered = (
    part[
        (part.P_BRAND != 'Brand#45')
        & ~part.P_TYPE.str.contains('^MEDIUM POLISHED')
        & part.P_SIZE.isin([49, 14, 23, 45, 19, 3, 36, 9])
    ]
    [['P_PARTKEY', 'P_BRAND', 'P_TYPE', 'P_SIZE']]
)

# 3) Narrow down partsupp
ps = partsupp[['PS_PARTKEY', 'PS_SUPPKEY']]

# 4) Inner-join parts → partsupp, then exclude unwanted suppliers via isin
merged = (
    part_filtered
    .merge(ps, left_on='P_PARTKEY', right_on='PS_PARTKEY')
    [['P_BRAND', 'P_TYPE', 'P_SIZE', 'PS_SUPPKEY']]
)
filtered = merged[~merged.PS_SUPPKEY.isin(sup_keys)]

# 5) Group, count unique suppliers, rename & sort in one pipeline
total = (
    filtered
    .groupby(['P_BRAND', 'P_TYPE', 'P_SIZE'], as_index=False, sort=False)
    ['PS_SUPPKEY']
    .nunique()
    .rename(columns={'PS_SUPPKEY': 'SUPPLIER_CNT'})
    .sort_values(
        by=['SUPPLIER_CNT', 'P_BRAND', 'P_TYPE', 'P_SIZE'],
        ascending=[False, True, True, True]
    )
)


CPU times: user 77.2 ms, sys: 20.3 ms, total: 97.5 ms
Wall time: 97.5 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q16/rewritten/o4_mini_high_small/checkpoints/post_cell_3_try_0.pickle